# 02.4 Indexing & Selection: Masks, Fancy Indexing, and the View–Copy Boundary

> **Prerequisites:** 02.1 (which selections view and which copy — this notebook is that
> boundary's *assignment* side) · 02.3 (the `means[inverse]` gather, promised its own
> treatment here)
> **What you'll learn:**
> - Predict whether a write through an index will land or vanish, from the desugaring of `a[i][j] = v`
> - Escalate by composing integer indices and writing once, instead of chaining selections
> - Count into duplicate targets correctly — `+=` vs `np.add.at` vs `bincount` — and know why the wrong one says "everything is 1"
> - Reach for `np.where`/`np.select`/`argpartition`/`descending=` as the selection toolkit, each on the collections lane
> - Prove a scatter write landed with a post-condition on the *target*, never on the index length
> **Level:** Beginner · **Series:** 02 NumPy & Vectorized Computing

> ⚡ **Monday 2026-08-31, 09:30** — the weekly collections review opens with two charts that
> cannot both be true: DSO has climbed for a month, and the escalation dashboard shows zero
> escalations over the same weeks — while the escalation job's own log reports "escalated 200
> invoices", green, every single day. The cause: one pair of square brackets too many.


## Concept
### Plain-English Explanation

02.1 established that selecting from an array either aliases the buffer (basic slices) or
copies it (masks and index lists). That was the *reading* side. This notebook is the writing
side, where the same boundary has sharper consequences: a write through a view lands in the
source; a write into a copy lands in a temporary that dies at the end of the line. numpy
raises for neither, because both are legal operations on legal objects — the only casualty
is your intent.

The treacherous part is that the failing spelling looks almost identical to a working one.
`escalated[mask] = True` writes 2,676 flags into the real array. Add one more selection —
`escalated[mask][top] = True` — and every flag lands in an unnamed copy instead, which is
garbage-collected before the next statement. The cold open's job did exactly this, logged
success from the length of its index list, and wrote nothing at all for a month.

### Technical Explanation

Two facts generate every behaviour below. **First, the desugaring**:
`a[i][j] = v` is not one operation but two — `a.__getitem__(i)` followed by
`.__setitem__(j, v)` on whatever the first call returned. If `i` is a basic slice, the
getitem returns a view and the chained write lands in `a`; if `i` is **advanced** — a
boolean mask or an integer array — the getitem returns a copy (02.1's census) and the
chained write vanishes with it. **Second, the single-step exception**: `a[i] = v` with
advanced `i` is *one* `__setitem__` call, and numpy implements it as a **scatter** directly
into `a`'s buffer — no copy is ever made. ⭐ **CRITICAL CONCEPT** — advanced indexing copies
on *read* and scatters on *write*; the vanishing act needs a read chained before a write.
The fix is therefore always the same shape: compose your indices first, then write once.

The scatter has one trap of its own. `c[idx] += 1` desugars to gather, add, scatter — so
when `idx` contains the same target twice, both additions start from the same gathered
value and the last write wins: ten increments to one slot store 1, not 10. True
accumulation into duplicate targets is what `np.add.at` (and, for plain counting,
`np.bincount`) exist for.

The numbers that matter: the INR lane's 116,888 invoices carry 2,676 with status
`overdue`; escalation capacity is 200 a day; the shipped chained write landed 0 of 200
flags while the composed write lands all 200, covering INR 254,805,215 of overdue money —
including a single 2,496,294-rupee invoice that nobody contacted.

### Mental Model

Count the bracket pairs between the array's *name* and the `=`. One pair: a single
setitem — it lands, mask or not. Two pairs with anything advanced in the first: a
read-then-write through a temporary — it vanishes. And a `+=` through repeated indices
counts each target once, not once per occurrence.


## How It Works

```text
  a[i][j] = v      desugars to      tmp = a.__getitem__(i);  tmp.__setitem__(j, v)

                     first bracket BASIC (slice)        first bracket ADVANCED (mask/ints)
                     tmp is a VIEW  (02.1)              tmp is a COPY  (02.1)
  chained write      lands in a                          vanishes with tmp
                     a[5:200][:50] = True  -> 50 set     a[over][:50] = True  -> 0 set

  a[i] = v          ONE __setitem__: numpy scatters straight into a's buffer
                     a[over] = True        -> 2,676 set   (no copy exists to swallow it)

  c[idx] += 1       gather c[idx] -> add 1 -> scatter back
                     duplicate targets: every write starts from the SAME gathered value,
                     last one wins ->  ten writes to slot 0 store 1
                     true accumulation:  np.add.at(c, idx, 1)   or  np.bincount
```

The table's four corners are the whole mechanism, and only one corner bites: a chained
write whose first bracket is advanced. The other three behave exactly as intent suggests —
which is why the bug survives review. `escalated[over][top] = True` reads as "among the
overdue, flag the top ones", and two of the three operations it performs do happen: the
copy is made, the copy is written. The statement is not a no-op; it is a fully successful
write to an object with no name, no owner and no future.

Note also what the desugaring implies about *detection*. The index list `top` has length
200 regardless of where the writes land, so any success metric derived from the indices —
log lines, counters, return values — reports success either way. The only honest witness
is the target array itself, re-read after the write. That observation becomes Stage C's
post-condition, and it is the difference between the job that logged 200 and the job that
wrote 200.


## Hands-On Build
### Stage A — from scratch

Collapsed (library-API notebook): the underlying mechanism — views versus copies over one
buffer — was Stage A's subject in 02.1, and indexing is numpy's own API surface; the raw
build would re-derive 02.1's `StridedView`.

### Stage B — idiomatic

First the asymmetry itself, all five spellings side by side on the collections lane, with
the flag counts each one actually produces.


In [1]:
# Load the committed lab module; it owns the INR-lane parse (M7 commas stripped,
# M6 casing lowered at the boundary with np.strings, per 02.2) and every experiment.
import importlib.util
import sys
from pathlib import Path

import numpy as np

LAB = Path.cwd() / "_lab" / "lab_02.4_indexing.py"
spec = importlib.util.spec_from_file_location("lab_02_4", LAB)
lab = importlib.util.module_from_spec(spec)
sys.modules["lab_02_4"] = lab
spec.loader.exec_module(lab)

d = lab.load_inr_lane()
over = lab.overdue_mask(d)
print(f"INR lane: {len(d['amount']):,} invoices, {int(over.sum()):,} status-overdue; "
      f"escalation capacity {lab.TOP_K}/day")
print(f"status arrives as {d['status'].dtype} and is lowercased once, at the boundary")

INR lane: 116,888 invoices, 2,676 status-overdue; escalation capacity 200/day
status arrives as StringDType() and is lowercased once, at the boundary


One load-bearing detail in that setup: the status column is lowercased *at the
boundary*, once — 02.2's `np.strings` fix for M6 applied where 01.2 said fixes belong. A
mask built as `status == "overdue"` downstream of this loader cannot be silently starved by
`Overdue` and `OVERDUE` variants; without the boundary fix, every count below
would be quietly wrong before indexing even entered the picture.

Now the five writes.


In [2]:
lab.assignment_asymmetry(d["amount"], over)

  expression                                    result   verdict
  a[5:200] = True (single, basic)                  195   landed
  a[5:200][:50] = True (chained, basic)             50   landed - the slice is a view (02.1)
  a[over] = True (single, advanced)              2,676   landed - a scatter
  a[over][:50] = True (chained, advanced)            0   VANISHED - wrote into a temporary copy
  c[idx] += 1, idx = ten zeros                       1   counted ONCE - gather/add/scatter, last write wins

  the rule: a[i][j] = v desugars to a.__getitem__(i).__setitem__(j, v).
  Everything hangs on what the FIRST call returns: a view chains the write
  through; a copy swallows it. 02.1's census already decided which is which -
  basic slicing views, advanced indexing copies. Assignment just collects the
  debt. A single a[sel] = v is ONE __setitem__ and always lands, even for masks.


Read the results column against the verdicts. The two single-setitem rows land — 195
flags for the slice, a 2,676-flag *scatter* for the mask: one bracket pair, one operation,
no temporary. The chained-basic row lands too, and for a reason worth savouring: the first
bracket returns a view, so the "temporary" aliases the real buffer and the write flows
through — 02.1's aliasing hazard working *for* you, for once. Then the chained-advanced row:
same shape of code, zero flags, because the mask's getitem manufactured a copy and the write
died with it. And the last row is the scatter's own footgun — ten increments through ten
duplicate indices store a single 1, since every gather saw the same starting value.

⚠️ Nothing in that cell raised, warned, or logged. Five spellings, three intents, two silent
lies — and the lie detector is one property: what did the *first* bracket return?

The duplicate-target row deserves its own experiment, because "counts through an index"
is how real aggregation code is written — and the wrong version is not slightly wrong.


In [3]:
lab.scatter_counts(d)

  per-customer invoice counts over 3,265 customers:
    counts[inverse] += 1 : total    3,265   max 1   <- every customer 'has' at most 1 invoice
    np.add.at(...)       : total  116,888   max 93
    np.bincount(inverse) : total  116,888   max 93   parity with add.at: True

  += through an index is gather -> add -> scatter: ten writes to one slot
  store one increment. np.add.at (and bincount, for counting) accumulate for
  real. The wrong version is not noisy-wrong, it is 'everything is 1' wrong.


The buffered `+=` reports every customer holding at most one invoice — total 3,265,
one per customer — while `np.add.at` and `np.bincount` agree on the truth: 116,888
invoices, up to 93 for the busiest customer. The failure is structural, not noisy:
`c[inverse] += 1` gathers one stale value per occurrence, adds one, and lets the last
scatter win, so *any* histogram, exposure rollup or per-account counter written this way
collapses to "everything is 1". `np.add.at` performs unbuffered accumulation; `bincount`
is the specialised (and faster) counting case — and it is exactly the idiom 02.3's
`customer_means` already used, now with its reason on the table.

The rest of the selection toolkit, on the same lane — including what a mask-versus-dates
cross-check turns up about the status column itself.


In [4]:
lab.selection_toolkit(d)

  status says overdue: 2,676 invoices; the calendar agrees on only 542
  (80% of status-overdue rows are not past due at 2026-08-31 - the status
   column is a projection, so time-based logic must derive from DATES)

  np.where(days_over > 30, 1.5% fee, 0): 21 invoices assessed, INR 154,496 total
  np.select reminder tiers over the calendar-past-due: final-notice=6, firm=86, gentle=450
    (first matching condition wins - order the bands from strictest down)

  top-200 of 2,676 overdue amounts:
    argsort  (full order) :    0.10 ms
    argpartition (no order):    0.02 ms   same set: True
    (timings machine-dependent; the point is O(n log n) vs O(n) - and that
     01.1's topk_flag used argsort when partition suffices for a queue)
  np.sort(..., descending=True)[:3] (numpy 2.5): INR 2,496,294, INR 2,332,361, INR 2,298,160


Four tools, one finding. **The finding first**: the status column and the calendar
disagree — 2,676 invoices carry `overdue`, but only 542 are past due at the export date;
80% of status-overdue rows are not late *yet*. The export's status is a projection of how
the invoice will end, not a statement about today (the generator resolves each invoice's
fate at creation). That is 01.2's discipline — test what a column claims — executed with
one `np.where` and one mask intersection, and it redirects every time-based rule below to
derive from **dates**, with status as intent only.

The tools: three-argument `np.where` builds piecewise values without a loop (21 invoices
past the 30-day mark, INR 154,496 of late fees); `np.select` expresses ordered bands —
first matching condition wins, so the strictest band is listed first — tiering the 542
calendar-past-due into final-notice/firm/gentle; `np.argpartition` finds the top-200 set in
O(n) where `argsort`'s full ordering costs O(n log n) — same set, asserted, and a direct
upgrade to 01.1's `topk_flag`, which sorted when a queue only needs membership; and numpy
2.5's `descending=True` finally spells reverse ordering without the `[::-1]` or negation
idioms (preserving NaN-last, per the pre-flight check).

### Stage C — production

The incident's deepest lesson is not "don't chain brackets" — it is that the job *proved
success from the wrong object*. The guarded escalation below composes indices, writes once,
and re-reads the target to prove the writes landed.


In [5]:
lab.contract_demo(d)

  [PASS] exactly 200 flags landed
  [PASS] flags landed on overdue rows only
  [PASS] idempotent re-run adds nothing
  [PASS] k > n_overdue caps cleanly

  and the shipped bug, expressed against the same post-condition:
    chained version landed 0 of 200 - the post-condition idea
    (recount from the TARGET array, never from the index length) is what
    separates 'logged 200' from 'wrote 200'


`escalate_top_overdue` is the corrected job plus a contract. Correction: build the
final integer indices first — `flatnonzero` for the overdue positions, `argpartition` for
the top-k among them, composed into one index array — then a single `__setitem__` scatter.
Contract: re-read `escalated[top]` from the target and require every flag present, raising
`WriteLostError` otherwise; the four PASS lines pin the count, the eligibility (flags land
on overdue rows only), idempotency, and the `k > n_overdue` cap. And the final block runs
the *shipped* spelling against the same post-condition: 0 of 200 landed — the recount from
the target is what separates "logged 200" from "wrote 200", at a cost of one gather.

## Evaluation

Assertion-shaped, as throughout this series (guide §2). The captured harness: the
**asymmetry table**, whose five result-counts are the behaviour spec for numpy's assignment
semantics and would flag any future change on upgrade; the **three-way scatter parity**
(`add.at` == `bincount`, with the buffered `+=` shown failing); the **argpartition
set-equality assertion** inside the toolkit (the speedup is claimed only after the sets
match); and the **four Stage C PASS lines** plus the shipped bug failing the same
post-condition. Deterministic throughout — a flipped line is a behaviour change, not noise.


## Design Patterns / Tradeoffs

**Boolean masks versus integer index arrays.** Masks are the natural spelling for
predicates — `status == "overdue"` — self-documenting, full-length, and composable with
`&`/`|`/`~`; their costs are one bool per row and no notion of order or multiplicity.
Integer arrays (from `flatnonzero`, `argsort`, `argpartition`) carry *order* and can
express "the top 200", but invite the duplicate-target trap and go stale the moment the
underlying array is reordered. The working pattern is the escalation job's: predicate as a
mask, ranking as integer indices *derived from* the mask, composed once, written once. Keep
masks at the semantic layer, integers at the mechanics layer, and never let either chain in
front of an assignment.

**`np.where`/`np.select` versus mask-assignment sequences.** A sequence of
`out[cond] = v` lines mutates in place — order-sensitive, later writes overwriting
earlier ones, and only expressible on a mutable target. `np.where` and `np.select` build
the result functionally in one expression: no mutation, explicit precedence (`select`'s
first-match-wins), and no chained-assignment surface at all. Prefer them for derived
*values* (fees, tiers); reserve mask-assignment for genuine state updates on an owned
array — and then apply Stage C's post-condition, because state updates are where vanished
writes cost money.

**`argsort` versus `argpartition` for top-k.** Full sorting answers questions nobody asked
when the consumer is a queue: membership needs the top-k *set*, not its internal order.
Partition is O(n), sort O(n log n) — on this lane fractions of a millisecond either way,
but the honest reason to care is composability at scale (02.7 measures where it matters)
and stating intent: `argpartition` says "I need a set", `argsort` says "I need an order".
When the queue is *worked* in order, partition the set first, then sort just the k.

**Recommendation for PayFlow:** loaders normalise text at the boundary; time logic derives
from dates with status as intent; state-flag jobs compose indices, write once, and re-read
the target as a post-condition; counters use `bincount`/`add.at`, never `+=` through an
index; and any `a[x][y] =` surviving review must justify why its first bracket is basic.


## Production Scenario
### Symptoms

**Monday 2026-08-31, 09:30.** The escalation job has run green daily since its "vectorized
cleanup" merged a month ago. The weekly collections review puts two dashboards side by side.

- DSO for the INR lane has climbed for four consecutive weeks; the biggest overdue accounts
  say no one has contacted them.
- The escalation dashboard — fed from the `escalated` flags in the shared state array —
  shows **zero** escalations in the same period. Not fewer: zero, flat.
- The job's own log disagrees with both: "escalated 200 invoices" every day, computed from
  `len(top_idx)` — the length of the index list it built.
- No exception, no warning in a month of runs. The cleanup PR passed review; its diff
  "simplified" index gymnastics into one readable line: `escalated[over][top] = True`.
- A spot check finds the single largest overdue invoice — INR 2,496,294 — has never
  received an escalation contact.


In [6]:
summary = lab.incident(d)

  INR lane: 116,888 invoices, 2,676 currently overdue; escalation capacity 200/day

  shipped:  escalated[over][top_idx] = True  -> escalated.sum() = 0
            the job logged 'escalated 200 invoices' from len(top_idx),
            green every day, and wrote 0 flags

  fixed:    fixed[over_idx[top_order]] = True -> fixed.sum() = 200
  the money the silent version left uncontacted: INR 254,805,215
  largest unworked overdue invoice: INR 2,496,294


### Diagnosis

Walking the ladder in its data-pipeline form, naming what each signal eliminated:

1. **Alert** — a human one: the review's two dashboards contradict each other, and the
   job's log contradicts both. Candidate causes: the dashboard reads the wrong array, the
   flags are written then cleared, or the writes never land.
2. **Job logs** — green, and *specifically misleading*: "escalated 200" derives from the
   index list's length, which is 200 whether or not any write lands. Logs eliminated as
   evidence of anything but the job running.
3. **Input-data checks** — the overdue mask is healthy: 2,676 rows, in line with history;
   the boundary-lowered status can't be starved by M6 casing. Inputs eliminated.
4. **State inspection** — read the target, not the log: `escalated.sum()` is 0 immediately
   after a run. Nothing clears the flags later; they are never set. The fault is inside the
   write statement itself.
5. **Version diff** — the cleanup PR replaced index composition with
   `escalated[over][top] = True`. The reproduction in the cell above needs three lines:
   shipped spelling, 0 flags; composed spelling, 200 flags, INR 254,805,215 of overdue
   money finally covered.
6. **Mechanism named** — `escalated[over]` is an advanced getitem: a copy (02.1). The
   `[top] = True` lands in that unnamed copy, which is garbage-collected at the end of the
   statement. Two successful operations, zero effect — and `len(top_idx)` was never
   evidence.

### Root Cause

The cleanup rewrote a compose-then-write scatter as a chained selection: the first bracket
(`[over]`, advanced) returns a temporary copy, so the assignment through the second bracket
wrote 200 flags into an object with no name and no future, every day for a month, while
success was logged from the index list's length — a quantity independent of whether any
write lands.

### Fix

**Mitigation now.** Re-run escalation with the composed spelling
(`fixed[over_idx[top_order]] = True` — one setitem) and work the backlog top-down; the
uncontacted top-200 alone covers INR 254,805,215.

**Permanent fix.** Stage C's `escalate_top_overdue`: index composition, a single scatter,
and a landed-writes post-condition that recounts from the *target* array and raises
`WriteLostError` on any shortfall — the chained spelling now fails loudly on its first run
instead of lying for a month.

### Prevention

- **Success metrics come from the target, never the selector.** Any job that mutates state
  proves it by re-reading the state; index lengths, loop counters and log lines are all
  satisfied by a vanished write.
- **Chained assignment is a review flag**: `a[x][y] =` with anything advanced in the first
  bracket is the bug's exact signature — allow it only with a comment proving the first
  bracket is basic.
- **The escalation dashboard alerts on zero.** Four weeks of exactly-zero from a job
  claiming 200/day is a contradiction a machine can catch; "no escalations" was read as
  "no bad invoices" (01.4's quiet-queue lesson, again).
- **Ship the asymmetry table as a regression test** — five one-line asserts pin the
  semantics this incident depends on across numpy upgrades.


## Common Pitfalls

⚠️ **`a[mask][idx] = value`.** The signature vanish: advanced getitem makes a copy, the
write dies with it, nothing raises. Compose indices and write through one bracket pair.

⚠️ **Proving a write from the index.** `len(top_idx)`, loop counters, "rows affected" logs —
all report 200 for a write that landed 0. Re-read the target; it is one gather.

**`c[idx] += 1` with duplicate targets.** Gather-add-scatter counts each slot once — the
result is "everything is 1", not a noisy count. `np.add.at` accumulates; `bincount` counts.

**Trusting a status column for time logic.** 80% of status-overdue rows are not past due at
the export date — the status is a projection. Derive time-based tiers and fees from dates;
use status as intent.

**Un-parenthesised mask algebra.** `&` and `|` bind tighter than comparisons:
`a > 5 & b < 3` parses as `a > (5 & b) < 3`. Parenthesise every comparison inside a
compound mask — the failure is sometimes a `ValueError`, sometimes a silently wrong mask.

**Reordering under a saved index.** Integer indices are positions, not identities: sort,
filter or re-load the array and yesterday's `top_idx` points at strangers. Recompute
indices after any reordering, or key by ids.

**Full `argsort` for a membership question.** Harmless at 2,676 rows, wasteful at 02.7's
scales — and it states the wrong intent. `argpartition` for sets, `argsort` for orders,
`descending=True` (2.5) instead of negation tricks when you do want the order reversed.


## Interview Questions

1. **Derive this.** Desugar `a[mask][idx] = v` and `a[mask] = v` into their dunder calls,
   and state precisely why one writes and the other cannot. *Answer shape:* the first is
   `__getitem__(mask)` — an advanced selection, hence a fresh copy — followed by
   `__setitem__` on that temporary; the second is a single `__setitem__(mask, v)`, which
   numpy implements as a scatter into the source buffer. The copy exists only in the
   chained form; no copy, no vanish.
2. **Design this.** A daily job must flag the top-k eligible rows in a shared state array,
   safely and provably. Design it. *Answer shape:* predicate as a mask; `flatnonzero` +
   `argpartition` composed into final integer indices; one setitem scatter; post-condition
   recounting from the target with a raise on shortfall; idempotency and k-cap tests;
   dashboard alert on impossible zeros.
3. **Debug this.** A mutation job logs success daily, but the state it mutates never
   changes and nothing raises. Walk it. *Answer shape:* establish the log's provenance —
   if success derives from selectors, it proves nothing; re-read the target immediately
   after a run to split "never written" from "written then cleared"; then inspect the write
   statement for a chained advanced selection; confirm by swapping in composed indices.
4. Why does `counts[inverse] += 1` under-count, and by what rule? *Answer shape:* it is
   gather → add → scatter: all occurrences of a duplicate target gather the same stale
   value, and the last scattered write wins, so each target advances by exactly one per
   statement regardless of multiplicity. `np.add.at` is the unbuffered form; `bincount`
   the counting special case.
5. `a[5:200][:50] = True` works and `a[over][:50] = True` does not. Reconcile. *Answer
   shape:* both are chained; the first bracket decides — a basic slice returns a view, so
   the chained write flows through to the buffer; an advanced selection returns a copy, so
   it doesn't. Same syntax, opposite outcomes, and 02.1's census is the lookup table.
6. When do you choose `np.select` over a sequence of masked assignments? *Answer shape:*
   when deriving values rather than mutating state: select is functional (no target to
   corrupt), makes precedence explicit via first-match-wins ordering, and leaves no
   chained-assignment surface; masked assignment remains for genuine in-place state with a
   landed-writes check.
7. Your top-k code uses `np.argsort(-scores)[:k]`. Give two upgrades and when each
   matters. *Answer shape:* `argpartition` when only membership matters — O(n) vs
   O(n log n), same set; `descending=True` (numpy 2.5) when order genuinely matters and
   the negation trick harms readability (and NaN placement); sort only the partitioned k
   when you need both.


## Key Takeaways

- `a[i][j] = v` is two operations, and the first bracket decides everything: basic → view →
  the write lands; advanced → copy → the write vanishes, silently.
- Single-step advanced assignment always lands — `a[mask] = v` is one setitem, a scatter —
  so the fix is mechanical: compose indices first, write through one bracket pair.
- Prove writes from the target, never the selector: index lengths and log lines report 200
  for a write that landed 0; one gather after the scatter is the honest witness.
- `+=` through duplicate indices stores one increment per target ("everything is 1");
  `np.add.at` accumulates, `bincount` counts — and their three-way parity is a cheap test.
- Status columns are claims, not facts: 80% of status-overdue rows weren't past due at the
  export date — derive time logic from dates and re-test what columns assert (01.2).
- `np.where`/`np.select` build derived values functionally with explicit precedence,
  removing the chained-assignment surface entirely; keep masked assignment for owned state.
- `argpartition` for sets, `argsort` for orders, `descending=` for spelled-out reversal —
  say what you mean and pay only for it.
- Chained assignment with an advanced first bracket is a reviewable signature; five
  asserted lines pin these semantics against future numpy upgrades.


## Related

**Backward**

- **02.1 The ndarray Memory Model** — the view/copy census this notebook turns into
  assignment consequences; the read-only canonical trick complements today's post-condition.
- **02.3 Broadcasting** — `means[inverse]` promised this treatment; `bincount` returns as
  the correct duplicate-target scatter, and both notebooks share the "plausible values,
  wrong structure" failure shape.
- **02.2 Numerical Dtypes & Promotion** — the boundary-lowered status column (M6) that
  keeps every mask here honest.
- **01.2 First Contact with the PayFlow Data Universe** — "test what a column claims",
  applied here to `status` versus the calendar.

**Forward**

- **02.5 Ufuncs, Reductions & Missing Data** — `np.add.at` generalises: ufunc methods
  (`reduce`, `accumulate`, `at`) and the axis semantics under every aggregate here.
- **02.7 Memory Layout & Performance** — where argpartition-vs-argsort and mask-vs-index
  choices start moving wall-clock, measured.
- **09.1 Data Cleaning & Pre-processing** — boundary normalisation (the lowercased status)
  as a systematic discipline rather than a per-loader fix.
